# CLEVR · полный прогон на двух T4

Подключите данные соревнования и приватный датасет из `clevr_kaggle_inputs.zip`. Выберите GPU T4 x2. В интерактивной сессии выполните ячейки подключения и проверки SigLIP и 3D. После успешной проверки запустите ячейку обучения. Если она упадёт, повторите только её в той же сессии Kaggle. Результаты и ссылки появятся в последней ячейке.

## Подключение данных и весов

In [ ]:
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import sys
import time
import zipfile

import torch

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert torch.cuda.device_count() == 2, "Settings → Accelerator → GPU T4 x2"
SCRATCH = Path("/kaggle/temp")
if not SCRATCH.is_dir() or not os.access(SCRATCH, os.W_OK):
    SCRATCH = Path("/tmp")
ROOT = Path(__import__("tempfile").mkdtemp(prefix="clevr_", dir=SCRATCH))
print(f"Scratch: {ROOT} ({shutil.disk_usage(ROOT).free / 2**30:.1f} GiB free)", flush=True)

for name in ("torch", "torchvision", "transformers", "safetensors", "catboost", "scipy", "pandas", "scikit-learn"):
    print(name, importlib.metadata.version(name))
from transformers import SiglipVisionModel

def extract_checked(archive, folder):
    with zipfile.ZipFile(archive) as source:
        for member in source.infolist():
            target = (folder / member.filename).resolve()
            if not target.is_relative_to(folder.resolve()):
                raise ValueError(f"Unsafe ZIP path: {member.filename}")
        source.extractall(folder)

def digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

manifests = sorted(INPUT.rglob("clevr_manifest.json"))
if not manifests:
    zip_candidates = sorted(INPUT.rglob("clevr_kaggle_inputs.zip"))
    if len(zip_candidates) != 1:
        raise FileNotFoundError("Add the private clevr_kaggle_inputs dataset in Input")
    extract_checked(zip_candidates[0], ROOT)
    manifests = [ROOT / "clevr_manifest.json"]
if len(manifests) != 1:
    raise ValueError("Attach exactly one version of the private weights and source dataset")
package = manifests[0].parent
inventory = json.loads(manifests[0].read_text())
for relative, info in inventory["files"].items():
    source = package / relative
    if source.stat().st_size != info["size"] or digest(source) != info["sha256"]:
        raise ValueError(f"Input integrity check failed: {relative}")

dataset_files = ("train.csv", "val.csv", "test.csv", "sample_submission.csv")
def locate_dataset(directory):
    return [p.parent for p in sorted(directory.rglob("train.csv"))
            if all((p.parent / name).is_file() for name in dataset_files)
            and all((p.parent / name).is_dir() for name in ("train", "val", "test"))]

datasets = locate_dataset(INPUT)
if not datasets:
    scratch_data = ROOT / "_data_archives"
    scratch_data.mkdir()
    for k, candidate in enumerate(sorted(INPUT.rglob("*.zip"))):
        with zipfile.ZipFile(candidate) as source:
            names = {Path(n).name for n in source.namelist()}
        if set(dataset_files) <= names:
            folder = scratch_data / str(k)
            folder.mkdir()
            extract_checked(candidate, folder)
    datasets = locate_dataset(scratch_data)
if len(datasets) != 1:
    raise ValueError("Add one competition dataset containing train/val/test folders and four CSV files")
DATA = datasets[0]
print(f"Verified {len(inventory['files'])} source/weight files. Competition data: {DATA}", flush=True)

## Проверка SigLIP и 3D перед обучением

In [ ]:
compatibility_files = {'scripts/research/kaggle_suite.py': 'eJzNPGtz2ziS3/0reJzaG9KhaUlOPI7utFUe25m4NpO4Yme27lQqFiVBEmOK5JCUH/F6f/v1AwDBh2xnbufqVDOORAKNRr/Q3WjAtu2TdJ3FohTWDL6IMiqjNLHKPIyS/ZswjuYhPtiPkoXIRTITVrGJoPEiza3yNrX+Fi6XsbB+ufhS+LZt7+xE6yzNSyvMl1mYF2JnkadrAB3HYoaACks2mIvfN/ptMtvkAL30F5tykwvd6GqVi3B+kabx2Z2Ybco0V/C/FmmivqcFw8nCchVHU9X5An6qJsVqU0ax/rWZZnk6E0Whn9zrr2W0Fur7tyhbRLHQs5rdDNTXZLPO7q2wsJJMPcrCZA4P4L9szhgVsyi794tMzKIwVniJuywq9WhpPltxY/qqGiXJzg4g5eOc/CgpRF46Pc8qytzBeTlBgIgFgesDudL4RjgutEUauu4Ow7sm1gTErwCYuwa+SujOjgWfk+OT92eedXL5m2edHl8de9a7s+OrL5/PLj3r/OPFlyvP+hgcX119ht+fvlzRg89nF58+w7+XJ5+o3d/Pzn95f3XpEcBZmCGPgzgtCg/ZuoiWwM6gyOKo9IDlMI8AKD8NpxE8iQS0WkR3Yh6U4TQWnvU1jRJoGKfhnCFmucBZeVYuZmk+BwKENwL/3sMfUW4yz6KuAFXMI5YweBTNruHvJhHBNCxnK8+6zZEIKDTejttFnjzNtNTNVmJ2HWQgdTAJeBGIO9CHGSAGCpDORYCQAZeFCElaJQ4eaw117BhiDjo2Ax7rYU7Prs5Orj4hddW7QIDKeSBIRQlEgTmHcazAqjYMOhZhDkgEM+C4yIPFpogq9r6jX0i0JbJD4Sm1jXsAG7K0CGONDoCQ+rieBQW0Egdz9Q6pn4DsbWalEmx4H+Rhch0lSy3YgPwmLAUyFYi1us/SciUKJFAWzq65ZwL25EZIvPXgv3z+9OUCCPGexAnE6vz06r0cKp/5GQhJHPB0dJ+TTx+IeCDbMZAui8NEzRH7MIowRxgnAmulND1Ec8cThL8p0mXn/fklMOL85PhD8P7s+PTSGlkP9gpMTxDaQ+v1wLP41xR+/XTwuHPy+dOF2TJd4otBH9oVUe37ouRfjzufPpxCY8eWM7fh/SzdJCV9Y/7hN6aP7e68O/71/MP5GY1AumBPw0LEUSIA4kMytPzBGzLDwPbEAuiPrDK21EJspQcbWv39w2pE9VMOq37qGQNoc8rwswE8KFfh7R8YQXbzNbylSNeizO8ZlsbcP0QE1rODOQ0P31HW8Hu/QgVVcx3dNbu+IdyJJdBas0R/J5YYcEDHWiAMVHtHBiZ9sMF2OpsFpKL0+icTuR7DVB87f9MLojUN3zus0OodVGip74xW7wDQAgH79PHd+S/Ie2cXWOtZu00RhUeGAdmtJLKGASAO6pGUeRrbahrw7zwPb0nuCHHoroQN1o6duVhYzLZgDdYudpJwLdwhgS1FCNYxB7zQRju0Slj7gD+rUxSv0o0oS+FnJYgw9ogWFva3/m1UCYPGUIIb27mIyc8I0sUC7Lo98b+JPA0chkFowJhs1xzZqw4fdKCiQTUCdcU1MppvwnjcmwCYJPE/gBqFuaPWO+uV1R8An18fMdAclpY84c6KIlEZoEKY1CiEmCOLWubjHwYu7hh7TBhuskRjfxOBHzXiBcxBIDwq2Xl43lg4HZte2KonmdcS7O7oKt8I7lqkm5xgKltfdcLRPYuadhPLEnEheAlzlPlB+af+dQa0xcIvU4fRehE70niuZEd6Dig9qGpE3Eps8HMblSv2ivwkDZY5dHKHNf1qcde/FdFyVY6HnjVUzJ34szS7DxwY28/SzLFVe78nm9uu+2KwCuqwLqJVN5xcUJQhLvngkeCw5LfB99E7WG8loUSWzlYFEKPfe4oth9Q4zUpoyaSA79HaP56H6787PCJ4SNAbrbDjgvOUj/xeD0wm4wx+wyy8x0e9vpQV8DLrwOIcVsKVmG9ikfsnaQHKcZyAgsBis/zw2YFGnsQX/i1DMLoJATxggLDsd0ktPNYyWxO9lrrQTD+miVDg2B9ELJtSiUAJGA4agfedltSzAUM6KXmUlYX2HkJw/wx3ZQ0P2H2TziP1nIoCab2XZOB2L5hgZZgT/SEy8MGLTkuAM5OMX0UFkPIeXo9Zx3FJJmLhVMG+LoXTV+R71XebponUzBAi9J1FUYFTIMnCAq4Qj4UImfr55N6Cth++3koTMmxgeBqqU4IznqIhH7MkIMUCCmqcb2MebGJqNyHxDUdmWzOpQUOsUQaMAICF05MDSS+W8a9rG4gXqxJpOBjFoEyDBPA37Js5Dkx6dn0b5nOn/k5aisTHUK/wZ3GUEUiAla+DTl154wNn8hy8x2iBQy6iBFz1jnERR1jwM6eNjij8MMtEMncWoPqlQyiCsw5rlOMahgU0rAkCxlqTA4l9YPVnACB5IJqJw8Bd97Fq32Jxnac/QMAsUJpvonRToNFJQdzRsFrg3M8t8MZZvoGXSMA98GIKAnn85YSlTMg43a8BhiY+Ms7XerkBJ9ppSU7IYkICqtW4TrJwM4P+SvccqXpKWNRAHoqtW/12xzZ0tOtCh+Qbo0UINNoBtYIB4N962wWN/Feaf51q+JFa3+xFfIOIU6FHS55uIVVfsR/RcU1eaStQGQDX0tbhL9aBNRpZvYZaQujq8JJNzSrDS1ZIL7JSDuoy0I1qFfs6HL3DkruwH7DBY8BmBF+CdX2w0RkBQcR/wNjy0PBb4WDLKcMj+c1r0euZjw2Lx6YUCNVpGtQ9nuP+Ye9ROaIUg2Bg6dCqEjBpDB8saDlUrwdyncMOHQuTCch0Xdo+VGvI73Ggqs7f60N1+xENp0lxkFwm7jcFZ16tmWZCRIkET7A+6R+sz2CqcD0B25Dm0TJKQLIu02kakxQUnOmDl5jt06ZF0ZSc9sKXTjOmaKTwypQPJXQc+QPx87RzVXP4tcH0Z9kG/rI5cZ+TLsRw1HQn/KUoJRYgCq6SJBU9G1z9l4uQk3nW3T2ZrQgUaDcsy7wAMzZN73Dt5rniQmlkvWrAQAHDtV0HCrHaUeUKcbx91CFglF6Dhh1pNgdhaGdtQ/4FLDNinZX3TiwSxoFNfDQvpDwh5yPPcvL0llI4EcwFJV8AdwSwXjjfoszsyg35gUy/UCfD9bgWIoPBiUDjqLLnMp8Fr2Dmc0w0E84yaehZUWVZMyByBk5qSTkuJHE0RYvCuiUcIvYYEMexPOto2B9MrF1rTDmlRopJ/Zy83I5lCvQEeW38GH/TPypniSTgO6BLOniW/+YnCDaqeKHi3RjJjGtcRyrLGRvsoAT8fPyV0flK/ih2lbz0kJLfYb5jsPjEcZWsq0jvuiZLpJiT+pmWwUh9SYvAE6qsl4e6+AIt0Ymzmp6oUdTL7TZIDuEa9tYz7D86SwG2bIxK6PcPt4yrc3hd477UKL+EId9jQbX502olc9VoHzxU73IFcrOCKHXkvzmUmtplxoAw2EcmPmK2IAvAAfxmdN8dpdPWv4MdxH//WsF23fGw/4bVnfRTGTQIrlFrCiRF/5Xcbhi+Ud+eV1wzX5NJUBNpcSkNAMr/CrQJAOlng6HZgn6NQ9W38mBZd6uMWJdQqKy8ZuBQ2Wxtsqsm32mxX7SmBE+uJlFsb3FyjGhVWq26aafhDMPNHZWv29hZ+HNtbs3KmuaXmVZZXINrXm0dMZYRqfNhnof3hY5kQJTB1QLX+5sR6KKt2+V5u0/4cErxv8eP04lT7ms4cWbq0ZNoKl+YXO5oBgGPtKQ1bt+EeRQmpeTZCzagaE+Xxabq0r0/c0I/T9UW1L/YbVJkrY/iGIkyipKV6zO0jkxrPLT6h49jOfvJSym/kJ56QPAf6K/pTCucZJZGcRb8V7GGuCcAKVmH5YhD4NkqTBKIAoI4LGSMdmdoLYAxaK0tv1oZiusRePNyAo/7DxWZHmU2FMUTYrtvDm3Zsvev2gdmez/JvoHG341o+JHGYMTqrNaCO9yqFnO1ONVMl23bFyJfAwNvRXgt5cGqtgvLVFYKCJmgXWHpQFKCnuyzQdskKNJYDfCS1dV/enXtsH7fG+PIKWgXmnZ2lJ9aWWOwEwGMhv/giPivGhW/y5G7bDUPYHoDRgIv3ZS8VuIiWWz3tp/MMvIar8XxZWv//7kTL+7AQuHGP23VkyGudsnRf9jqOm8z5BPFUeknUJNqvBjiVU5INXwRzTzpjxBvyScZWX7/tZGWg4geWaJANbKk4Xx+h7ldFgrEQDWsp6LmUSG5HGMAvfQx5+g41B19DMzTTKw9Js2Yfrm4PvYHPVgtezDNEGg32uvXk2WJQDcCgfvrKME0EiYoozAecXraBR/rp1qP2SotBG4iKTzHAGMyHq7DO6wfGRzt0WyRUa5bn8MUJ0CSLInNsHj7pJH4nOKkDC+NKZQZvdrN9w6GqiGydrzXn2xrb8gMTBTrg0BQExRWpyq6mDZ2bWhSHT2kVDr+G3QCm3i6lCh2aHsWWrgNoMQwBnrDHoIjxbsSCQWpkeiUTvPY6Bp6VlLrwdLu8R4+KqJUGkUkg0VgcpZ5uCaQLwjyqiwldKhKMRwFRgeBTyqtCpUgluu9PXS1tpD8NPZc8COtn3ITTavjWau62jnRq75r/cXqD970tmVCF3aFulwYfV4YQRVegT5U5hQfo19s5EmfBdHu/SQ86ZvJScrFVBbB/JlpJd6OQaXuXrh3WFLFIsByMWhn7K0ieuAV3Nsqc1wNgkSX27zGljz+xl2yERXtONncPwU5e4fbJs6DHYdTgeUShgTSo0ekFKFZ8RgdKH6mH0lHxtHI0ma2owb1qNZGzkq6fyBjFXY/6CIjK03iezCQmKJEpi0QQfc/LMKmAMdZwMsbkVtYQYhaoSsYq60NlM4XZcIUfaWJRz9Sz8BtCL+jqyskkxTGUhnURp7yCII4umaCmLZHwZIFGk+D4s5KHkHxwNVsS1SxCvP5n5c173CujFCXNqGagRO1bYVNVAjwTPa7UTKgXcVAe4oBkiVO0e4ENPesSjrUHEcE0u03cl3ZMk83GVVugd28p6IZQUU003hDW9LLXAgq3ZoCV+hLtskhqKQKr/uQntyLOE5vpTcPjKUSGpAAiu+cMZeywYo+F3fOjFeFGQqtgcJE2qDlNJguDUFUIYGN26LYMpmTvsutogQmqhbSkLejSRC6TN2g2hPQ9X5ibqXTrxCQFbiHSN5+gbZjBqEOblSDPbJm8AcLhQUVv6qAIE9jXyV8UMy0H+f7/tZkT0WTp1MH7PClYXkweMZ9lpmWhSVLfh1YMUGVft/A4iALMow9+TDr2HuPkBXYrb5CAStwXkRC+G56wiSOVJDIGZpajaIDjT2sJfZnN+UJvqiekCQEn3/5efD+8jdgCbO7UZ4CSKoVVicuGLRhQXSGYwz0HyP2QDUnAxpzLKgesWrwL6D9Ej1F8EM9FtMxK1HrdWfyRiODGBqui7FuSqlydtubx2Wr3KCkMIDmAfj0D3+yXlmYU0VkQJCeTpbiXnsxQp+5T34LMZxdFucAnL1Bz5XCQUUUlApSBcPgNBzMa4E4/ECtQafBeQu9D1971sEAkQL/fTsicbiezkMrGWrpkwo5rtQxQScIlVG+AtpyuVYzzVxs4rKeufta1Zj0vBoMj2dliHbG+qHG/zr8+oqamGzCEZRkjct6NFlxxFQkd1Lz6b6SI4eb3M7X/X1GAWZ3pB/LQTEOM9HtdP22s+DrK+xM4zenbbhrHVmTg3ktXRJIkDJtIqGM5L8Nru7u0kqEXrpjyziManejspQrggAakd0PF6XIbUDHaUQjZoUE5ReZ6K7rthZw3tD5M93K/6dLt95v+L7FGuxVlISlWuw43eLsghCghcPVrb1q5GIR8baj0d8MeJmlEBchQ0n5PFMBSXyoxKOHdqYSY0qNNZfop+XPxcWTxaGuDhGlk/jN2JbC2aiFMdAfQ/uJp6ZGv2rdWXCpBT+SAtwI2WnivrgrOd/PTaWET+oLEhGn2ZRVYKJcistZGIPTvrAO9k4tdlP0lmFhzaPFAtPNfBAGHYgQs1N7tKpVMYnct6vVG9RLnLv23nAHxaBPcytFJ94VOKqR3gZIknULkO5amybXo3W4FFQkpytvND01g5rlYEx63Gm1JQ6tJkRy11iN7AWeW0G1jdZgtW+MPo72tLiX9Z+V76WGYqg1gLSqApAxL6ETwqZI440snR+PX4MmHOKLMa6Rbw8n0lXQJT18vEYV/jvPmjV2H9ld4GJCZzweHw58TErAn8Fr/HPk45D9Pj6mdwdH+OfAnwACEvSolljWKdz7Nnj/7RvP8o/ebOlLFqXdCxNsW3qQG9XuAd7J6y091mFxXege0prJjAfHC27DO3a3w8H8G47VGx4eIX2GR4doFPr1Fq/pFU4d6Djsv+lVbWgTjQ2F1zKKpmMH3PJM2koHU0ZjHg/Gcb7poVXV42lphXEs3cOo4JJM58718anrY5WddEbuyPBux8ylajuWb6r8I6n2w+TerCMH2wIu4edNgpHDGRaDOjbaKGNSeLQKVqFFjAXV1iKMYlAkKQmgLrZ+B5q86OyMsRIEUon1wKzBjR3WBrYYs+LG4XIOD809HoFbRiWdLMuXQu3pSfJwQzDlYLOBQsEmiX7fYPUWu0LzwsXTFhTeiZJ+EyHUD3yJ300wLXr8hpUZkhqX1JAivRLrJM9PC2u9gS9T2uvBsfHdGh07SRYMmLO5fyly3B+hE48OT8klN0vcjRAVPxcc89ZwMeeKeW4lA5mUAZwMvsv8qShvIfjmxZdfPjmRzXQdFQWz09jA0bPhgWg2IFqoz/2J3XC/GVXTR5ArHrMKVSbzsYLZ2ITRARA21AdKElhBvgmFMsNlumHOimQCj2XS+RpZ6qMn4MNbiRmP66Fzck16cU1V2gwOyHhNZ2+QslUszJhI79KkmVDHfDvxWNi8jj3S8B5mBECti5FknB/NmzPHjy45e6hKmengmqqk5MNODaenqrRUh5PaZbuyIIoO2CBrNmtHPqJ4dZePQNBgHPXLBwZQ7kxmpXAaKXw23GAYko2ovah5d3wU1iyqVUTSXl3YRr2Sb1pzA/H7JoydEClZsHMmOeG2+3ZK98L+NSpIB8HSoMsIk3xQgRPhY7dn1yAgDE9hh90Ru0t0VNHWjlzWMNQItkUe8gwI23K2fI4NUaVy+9nn195pwR2kAUAzWHJdtn6Nv+1qY3O7hNSKUU0saUSecB2Osrl0uAKGUSJhRtCUXD+5/M2sv0VN0E0qBYWGLzPtHX1RibAbJ8jZXFKmymPnTRUf2H/x+4dLg625AIf5vqG79SQ17uBgIR835ZUCfzOK2rbya1RoEsxCvocHjSbK7D1ti7dILJLyBtaIRTSj/Wy5thJPHswCCJmpNyKvrcZDG2p0rUlDwW9qFb6z4+0ylUoIXUzdx9FUGuTBZukB35alZHf34XqobP71pLK4Dh1RwOC/DGIRFmXw9u1b/H2LxzB4F912Hx+l47BZr8Nc8qraYcGhXR+cwzJgPVNQQ0z2z/HUn1EUKoEoedGztGH8ucinKQSX0lQbUqT8X4psKgh4Ri1ZOmZDlLdNsZIHYnjdCkGXoxtAbJHGMIZkMqWoycbbbGOpwF/dDsBBC1YS0WN5iQA+5SwHPpQ3CuBDxBie4MUE8vSLqaQsKqSnPGpbU+dsNhhD4vosFjd5IHn/LcoqlSVbLi968P87yt5FVLOHe6I2pkOw2AVQRL0c6WbnF8Hp2bsPx1dnp1ULGELEowO2+EylugbgLBT+iL6eS9vE85JVrvjAVQ7OraNa+/kyTqeOvWu7W9YG0HG+N6KgWyKaimh+JJ4+2SppcKgvl5jciKBM9cjmaSb0e5GO6PE+ILl4U9X6h8W/MAEE7jpoQQFODvBgsLs76A39weLR+jX6Wfu/86jIKOUKZkTlAbEkChQNnUh8Ss++plMSryoUxbsh7GF1RQQdTtEnU8wyOTzxTEVl8LZx1YLa/2xcsuCZw6RTPOmtUrkwy2ZrucxQywD3JVlhDCCYt8LQWx4iVofrEQfEWH33+JA/PFKnFUhxZnxEvX4tRAUcD8LD6/pVFTZms/W8+DILm++vwAM9tYss1Pl1uZVtgOZcJL4ydhU99TzQ2lv7jTtgMF5AgT4iVt2nUUFW8X+jlXoscwYqFECJIIeGJ1tNReNiPxoLNdaWUK67dOi7q0VojJAmzm6m5A/kdCloY0apiUzA4HLjsJBpMfGsOzP0lPW1xj0OdHxD7nwQZFq1NGLyVDZ0zjbUQY5liAq5QPLWnaqdgQm7VC/DowZX+YZKauW+BwhTNOWDPGogFHjPStzqegl91QD5rRRQUAb4mRFU4o4BKxrqWxEwP15/ChI4b+KdgocQh9mzUPFaBjDzYdKAW13X0AAs6wKZJRjzGZSn1LOkvLQdnlUY93Nw+pfJr4ci77U+yCJPv4lEnYwlVZdgySbUydw8ylQHtUk6gGFSYZrCQoFXMDFg/GnKrEzW14HxrGH1LMEFWqMzB5oL5pIXpMbEm6CQvmRttTxW5Zv0UtosrzEoKq+uLeIxlD6rEfCGo0gWRkVVZr0Oh07JkMGz0Kxgahj5txQJlTQaS0CVO9YVrFJs0Dx6tZnoSVZzadAMzaXVMYnKGH3XNGA5oGS3KlCKCjLB4LLQrjqOI+WlWr/HuiZE3dAAK2UlTaZdqE4oTtS4xs4FhWwkvpKDHFtVVlTVCnGZEL6suxFs33Spl11dEdZhtuTC8UKzVfkZWkDJGktOqVVIFYQTFystlW+l1L94PnqoV9vHUvLdGkyZhWo4SR65KzKuDeXoREo1qrHNhZkV+bixtWx03K5LEqcXiGAD5PPy/YdBg3eZs7ltp0d5DLmM/y/GIO8jV0YQ/SgJTJok02WpvTFqrDq28nfHbX6/aG3sBlU5fp3QnK6bfozbZCYmASa19CJLnPRrMOUCMk4V3uiQ494XuvA6kyxPLmzmoczMBKSHDqUEBk/nxy/pvL119dq6GyDS8hbAS0H30hQy1Y6drOM4NhI2KqRx5N1z6lo5FSi27pejy+na+OOHwpT19TzKuR41SK/J78YABi/CK4x7HMyMgdO6rYhMDz8DlSuqn5VlMg+LHNkd9R61FnSKQT1JUJMgGKE3g75x3Zd5vIouKKSs8sA4o1Kd8fasLWe/9fwMJTGIRBfbjUxuLwVeCkMcx7cipzMMUSO2w33p25CqGRY2+BbWQ/Q4tB4IHod6nvpVpiVMj8/FcKB3IAO9XyjQk3gsMUYGVJjnmKVQD+1aky6eqnredUSZFMrDVdkOfs4pHQlKnlOvZYGM7rW8D2mD8Y5GLhxZ3yyT1hwnyMsL6CwkD0GnzW1WPdzIfTRRJe/fwKA9dEecr+kgw3yZqpJLy2ITx8GTZ0UwlsFkuBHYVDKCyR2sSqGAFWOh5mkQ7FAXId4vmIkoKx3s0UzrybOXisEknjTOsDeYP+49/Lj3o4/b6NxZZt52jHzdWlccU3TGhUYy4CMUVbK8hoVmU22F52YKIrpRSu1rDGx4MijwmNFbs7gz+o/7D1RlBARxKa/BCY4wRgbeU9YH701tZNVb+wVqIwO9jpaLVMdDQJhT8JViTc5aewbnXSDyYa/WFf0ZzrR0byBRRru1aYN1CLRvg1/8hDfzGkc96H6V7YDxhpY/BLeggpCRgfm+HqshYLRwizLUUi25SJe/PO1aYE0pjVTfXFJ8UFAJl13LaYNvhOKTtuXv+ryivSjeeaKKdI9L52rwVd6ScvLYyCfuFpiRJK/cspvHUUhSz66OSUxRIKwc1th/PkjJeVWb2NDvgQ2GwUB47W14L0g0iHT/fKhIbfbN081yZVFmKUtjmajnQh2IWArOjdqN9YNm+qw2mb2INE9aNvxknAWnk3goVFrZ3Z2axcLCIWeZbRrW6nYVAaclkHZatMzvu3OlmIdEDeCOeI9cLBZlQ6TxI+5mIiutc5wzeUtbtu/IarZercP8WpVUkKmjrGirGS/fiJDd8/o23W4EDel6Q3hGCQBevdCLRiq0QMgJUbf+cPIECJ2CrU0zucFs7O5uCiFOchPlePGDffLl9Dj47fzy/OcPZ8Hp2W/nJ2eXlHMldKHBp18vgo9ffg2u3n9Gbxbe2QN0iH7924eO59tk1v50cfbx5w/Hl52wLv7r6v2nj18+/vzl3buzz2en+KJvP7aAgaits6dETTdEeQYiqlw3TYZk+EdLLm3Epq4N1uqqaR801Rnjnc6C7rLmaxjsvTW7m3xVHeZgMCXvmye3scHeHi56NtabwVDPH50Hnozgf4/Tqh03muFH3z9GsIFMlm1MB482wKRN9j1zd9M6o7ubWuMYVXEs4J5VvyxLfaTXVL9Rq75T074S3BnQjgsmguq6FvOp3zG+8Wl3tXTQLOA1AK2wsrVRw1eR0zZNayOH2tB7nzcCDal5YhHfM5bwfWMBr686ajO+k8TKRNb7Nwyu3onpADz0+9qwK1TV8qHfmZdwdlQ01hxuvDZM9g8q+ZBPULTZNR7WZ4n5GXTtcIeCmS4d56osZufFw0uYAYedAe9pYYK4S07rPHj5BTjfOxG9T2pGrzuaXdp9RI6dJfO9Mt0DmbduQ1jUEWnr4TncTXZx0L+muyTlHS54MDRnE0/f/eN8CR5uUl7QG8SLrA5ubAbBPJ0FgWv09MP5HEjJXZzKACVoHEb2K/vJxummBH8Jaz/uMzG6oH1FwDAEXaFfjr3PJm7/Ns1x00mVhKQZ7+6OFGT6B2EXjq4Mk418RMkIVNRWovm66xShmRtRbRlhJCTAD6hQNwhoKQwCJGsQSGedabzzP4qItMs=', 'scripts/research/kaggle_suite_crops.py': 'eJzVWl1v28jVvvevmHLxFsOYpi3HcRzj1aLBItku0O4GSXZ7IQgELY4kxhTJJSlbTpD/3uecMySHlJxk26JAfWFJ5PmaM+d7xvO811Xx0eQqzhO1THNz0mxzk6hFVZSqMmVlapM3cZMWea2KXKUJfqaLOFOJacyiKSpVArSo46wOPc87Oko3ZVE1arVovzXpxnSP8+2mfFBxrfKye19Ui/XRsio2qr7NTFzlYWIWBV7WKfFVFu7NDy8FihHap3nuPAzzDnq5zReEDUnB7fWRQN3Gq1VmonqbNiYCj01P/oeXP/z1VaBev3r5/te3r94F6qef3/z6PlA/Ry/fv3+L329fvfnlLR7849VPP/71/buvEow3ZaAWRb5MV9sKIGWWNgF0vDNJ1MQ3mQlUVsRJoOr4Dt9r02yBAOXe4j82IbqJm8U6UPcV0f5QF3alOXbjDsywEaaqW24/vv3l1zfvjo6OErNUTF7n8YboEl//+kjh7z5t1tB8SHw1L1idqqX3iSA/R58Y9HOYlx89n7SWxE0siPRXQcAqh2glaNOr2e1cLWEAtyrNlfZ2XqC83QP9v4uzNPF834oTL0RiGEqtd4Hi11YkS5UfqT8rvZuFYRios7n6fqpOJuFlS8TsmgqEohIKzGpt1VlDDaYO1E2xo486/YglJ+YuXRjLQODVVBQRppt4ZYBu8SwoQ4IGwMSQSNMRG6tm0n7YFNoFjncpEdUCHldxvjLaZT+VD/xsHkoztWSh+ebpua+OVfjMh/IJg+k9PARq1/PfmHq9qtJEE59Ayf80T8wuzVdTL/3giRgEAyTIOLuGrRa5af9fn8/BRajVTby41XoH3T88+BAp3UxPJr56cgjx/Hru7szrkHhENcwZ+y7aDJhvuNlm+twP6+2NnoAqtnCVwwmqHHY5fY2IAGplnCQQOdoUiZl6N0WVmMpr93RdrCxFu1mrKn4grdpNe2J3L8zNfdSYvC4qPQvPX7wIoL6r5/g/mVzM/ZldsbuOOcm1gVwt3SSFw0Sk4dchhNLEipZ+zStWJ8p5cH1yPg+UngRq4o8IPOwRGGEL6hlMGNg9AZhdnjbbxHRbvH4oi0b3kgUOE8Fp1qaJO3h4XH7ewz848Duf9+IKBiWwZeqTrUFdPpw9X2lf/Z+6YqILk2X7lnt+0dmtVbk1dnV6qi79ISZ/ac1mjl26gKXxQ+IaN9gpLSh1VjS0AFpI92pC5n8FNEZhuHVaE5yz2x9NVdRaZ4An1cK8JudXPjZ1QUSqCGYV0QYRh6DXrsPEdynTR3iXmnt9QptyGagr37VyzXCn8smWQ3C3xpTkK++rrWGfPbvwfYeFNeO7tEamYRPPtLVkSUpQbo0QuXEC9bt0laXlb4zyd8I4cuXYeyuRCHkYtJCeE81pCZJ6NYOeR8Ldo2yCvBwt08zUUZFnDyK2lZGCb7Q0Mbgg/LGogRr4HicHeLkeGkCfAEy+ABqFG1kpw0WI3FtTW6tpdZ6gTACcxQizuG4ieRohFjUS8uqUvSHNGy3vwnodl2Y2gUk9ofhoFZNsF8z1dRgnccmpJL5bRWVRZOdJi8qqRslgyCgQlLBKIsYGZGULFO0pcZX/eKCvyBAGltDKTORhZ8W2KbewMCtGt/lgMWJpFV1vS1NhT5DlEa+hwS5XtZmvz1pkvPUoX6VLm6koKas/TZXHO+/tJWLk8eU2y8RHhCJWdMHLPEPwwXuodmKdV6O2iLDjUtAF6gmWURFCaxN4DR075Ym2jAMytcwb5soVOVWLKin76npyLpmjiauVaShU/DEZqZiAecBJUFBIXALMEPXsxaVjknVGiT2DWJoRA4t/LIAdXBqoDwTJZGZ1Nre6p68d0O9bU1F4380IHFHi2NZ+18/ab3NnE5YAXTWztH+WFfey5g3y9QYxhCn2+fWaUgMQRV/41Qu4Tldri5vmj+BiVaOHSD1DggDpfuKlowAuFg2XxBTCid8J5PXDBYKIPvNDlPEJgl6Pss0Fdp9nBzti1z8/GTDstwvFganF53txToWVSDIxJ1f+PgIUc782FQoQMWConarFkj6/V+HVpR9YYCReB99kYMLRQ94i362wO9pdZ03W3kPIayo/w2fPOiiyzRVqHHhNerNtDJmoQYFoKoQzLc7U2yX9QRk3JHqHQmbVCjQfgFqHgTUiqc/d5VrhaKlEzhXuyg+ER78mcatBOLOk3SxgK2kJS5HtEtZx1ZbkUevqpBPqSvTzc6svRsHjUVszIOWGiR0HHSnup6oNK+v43rONCeNY2m5QFB3kRU51gD7cQNiqZF0gBAt8W4bTYkaB4/wwExsRhEgXFezPNh7nKGIcYk4kjhHn1W+UAV9VFaot9FL9sj5fA1Mt0FmnaJdgxjAZ7qtZPvWJPz57XfPBhr6bjZLDo1Eoi29sY/MvZxxpkODCKdXFfUPIio7cpUSttIzynXq3qKg7VVxtqLgy6Cnq25Mb9BlIkupnc2cqMkEkEPQFGdoAgm6KWxTxpIe3L/+u4FG/4TOU+phKCggxrKX6rutAHzW5pM2R1yG9VFNky8U2iT0FzRg17LlCA13YuvQ+TZp1W8iEYs1hW6C0LRlXAIlYVpbehFTIxU1YlADamM0mLpF0Ki368489YK7QPFMHLN3O/bHXik2JUGRmX8OTYUpkgaxFdx3rt/JFG/Wv8uWiWpK27M23cxWEb2A8CHWP/x0S7/JiqBpQpWnJx6EoPLFobXw6MPWpNXhxlql82FBAlQLvL02owk0BNy/ydAET4fd/EfNJ8yUCcb4wbJPWfnggwYWiTp14wN49/eKoAgnLTiv42/mzy84bh/30uNw7WMDTQysuT4s4hNgJTbpouHm4NQ+AT3OdD0Kizykt59kNrCBQLy5F30/P2wbJKp0pM912KKVtiDh1Q8RpGyKCVpZHdz6LNzdJrPLrVol97M6pU3XFHAarpmhiChOumaAKERFP0G1Tv8q/+p7m8D72uyZlpymDvvjsM/vhMpQ5kPuMcn46zENSkpLu+euxnei5y/MH+DFId30TTGtYUHA8YrMR36PiB/VFuCi3CJUyrSLhhg+GNNgkBZGmL3/EVLsRAXrgxzlQ/wJlil0FsmG+sirG3nKlYp/TZj3zKWyfXe/ZCk1Cv2Zosm1MK2gd2neCN2LQtl5rN8INHtlWt300nDSuTZy0jjZo0b5Tv2QJ8tt7t72/r+KypkmHTWF2EytKhvecDDMT13BQ20EqtHhuoYYqjSrFlqHnJkIbYDM/JJn6pPjFGaNdDSlCljJYAz1xQhUDDHPvqOFbIE1mVLzqQX1JiG1xuTaLW27L9dcryf92vu/iQF5ENDdzA4DYStCnwIOxVtCx5YlGhHxqY7cTwKcH47jfVpG2nEhrbqTII6iutMGpXqLtQ7gRID9EzaT9R2BETAszLkbfbnNKZ1KOeu/S1d9+eiN5idJlhravoa5BZippLqW52dwYHtPWtho9ZB1DTwh5ONClw0w2dFBcsa1Sh2LsYOdkYivXNDdxRaEyD//G3zXjdIcte9P2omy6OSi+p5vwZRJv/qFnT4QJAu7GUEtJZvtE6A+eIn5l1TQ8OzubdJ1AnJmqn65uyvBH2MU7fqzFrAI4MfUsyfSg1dFEHkGJKU2RQ0PH2GiQtjdB69pZkVCzXruwBLPVbULIirruW1L7Lqx/36LUxpeNibsRKwsc8ocmND+kKvweEXIIsc35S6ShwM4oD5gX0yDTc/aPPOYrljuC/iPmSUYP4ztkoXHeGSmrhLpwO/X2hutHJuhX1i65pMarjfUVTRqX3qxjM1euf1hjRzQKOonoVJTtDT2BzWBlXKPTorPQT7K/lIc4gQzmrXTEFnGeslm1nQtTJuT+jxLk2bCE6FoxKm8Pt2N9J9an3cGxntCwRTEf48WjGiVfFmBB9aGOeyIPqckSK0HAQO1hIjn6IKxX+erxwG5Dx17aGQSMAxHgULT4XwsIdiIRIxOCzkx3e7LXLqGdskdileeLumeedCfeXErzUh6TnbjGZKeyvoRSs+NzuZrZfQCG4OKbjtquxx9WtCyegKX0hju1uGlnLdrOFb5HUQYnzh8oh+0xoyK3jqsqftDt00FMoRK3e/GFgYn3c9EO4HhdJ3RwIf5oJ1etm3Np/livJh5X1xFWpavi3uX5nXqP7ag3NIaQPmFRlKmh6wxgRQUblWZ3cd7IJLm2Zzd4QdsU9t7p9OVyojpjXc4+zGdnc6qprepZrSSFM4Btqm2zPow7+RruwWNpe6jxxYrp6bkjAKvzECUWzfUwe2Q4jC8HE9qjSW3nNDY2l50NGq5bOq9apU09NM5xwuPp4oxPAcVkVwQvlx3mNO70R9Iws2M6LIIRwSCQKGBMD1qYBVYNNM6+BX66ysEv4oP1qRxJ8EEPZYGpV283TpS1BS8zOLV0QuoN5aSZJunQUUTd3sQxSw5paGuvB/pgQ20dZHag/e2cZ76Xz238C8ldpZZFCEZ8ieDCpk1CBPTNI4GOmTsUeHp1YctbHg9cXnxhHmAPIzlLOP2/LH7YSBADUxZ0g6c95EE7+GzUj6fJzsYmDjJ2zIzEE5am2mzlRpIeyh60veXwuXTdsJTTU2nbRxbzNVUObX1UbA33EVLP/YP75SIPKpXRy0HFwm5LXfBIvay+QF3YltcGyPhOirBADcCdt+Kg/fvuZKINsv1VIy23nfpzZiYWcYwOCQBifPI2vM+JPJbxvHc9tChwY3HpxcW3jgLdPw92yiJeKz0O/iey/tPLs89d885dd8S32LpxxrcdbnxlED6agXM+j8qYo7qghhQoeUZhqzA5wz1Wg+xvCxg+V/5mfGzCsnHQu9K9FSI0u7RuUNhIzdpTR4XOdwIGtTiHsv2xb9G2I4w5LlO+XN453TaXYFKz0GF/xKUmE7d350ZmFZaN186+sZ8pT/jcJbBe6u0SKtFe2GzKgRK/OLO2BP/TE2rbg7czg75k9fcGDuPBY3A4lT5++N3xQjAenzHaomgvpct8kOldy+wRyPNwQanQ//aioVfygBZfqiDPm9s2X36F6zhb6kPDQaExGL3RzKC9XCHiugYQVqbM4oXRjhn4I79DR4uIdtvN7bqBjXX7u7hK45zvXDYNXZvjK2lWg+hxdg9fPJq0gbPYVgup5m1B/sXqXDDndins273NL71PVqbPUbmIuUNrvdm+4Asf62LlXPfAS7uAUd3FEZYkGx4mI0mKR9CoqtiEUE28zZoIz/Xz8ebyUmzPF9FiZL3709gqvneChG0yl17UrehgZ7NHRoQO4xIummgQnVFOX6wLusJBZoVHvozMr87OrAPQM58qMzYJu41zNCBkutpbeqMpOqcjkXZR5AuEoJxrS+Y9BE2QUfKabzdPmSvXOux1RIRuMtCwmtNeeztpb97tEPl/NTmku3HX85I6JMivmvvCisuphk9SK/P7NkUZypvz5oeXIzW2k1O80XTdclOiVsmbetqLEaj7NTJ5zhUMzPkO9llkd6bCprBRpB9Ngs2SHxKlp3u2IYejsD2ri8HL7iTOZgoT51NBoK8R3XzuBGtPWdsH0SMBVwZpRPn3qpF5I9IaNp0uvkViaAijfndT5Jmz8cMhCIULnn0grKV3I3smCV0BA+FMrmThR1ebCYHuQfUofCuKkNpG3E1F9u7iEw4uYjZy0hqoNum0unb9r40nYxf8N12PWNyYVZrvJRbxtavJi/PDaUUuVjFuICSOGXgAK/f/AE3OTIgDxxy7yuNhbo+e1vL1hLTvq784GxZSYcrqt7Zw8jxQz4fM6ETgUepPpuriUJKTOYwcf3lzWU47neGT5P7htZDqMz/lQnnmno+MYhDdXhFefFnHd7LQ0T8BDdVuog=='}

import base64
import subprocess
import zlib

# Stage the checked sources in scratch. The attached dataset can be read-only.
worker_source = ROOT / "worker_source"
for relative in inventory["files"]:
    if relative.endswith(".py"):
        destination = worker_source / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(package / relative, destination)
for relative, encoded in compatibility_files.items():
    (worker_source / relative).write_bytes(zlib.decompress(base64.b64decode(encoded)))

env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", OMP_NUM_THREADS="2",
           MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", PYTHONUNBUFFERED="1",
           CLEVR_DATA=str(DATA), CLEVR_WEIGHTS=str(package / "weights_in"))
env["PYTHONPATH"] = os.pathsep.join(filter(None, (str(worker_source), env.get("PYTHONPATH"))))
worker = [sys.executable, "-m", "scripts.research.kaggle_suite"]
from concurrent.futures import ThreadPoolExecutor

def check(task, gpu):
    subprocess.run([*worker, "--task", task], cwd=ROOT,
                   env={**env, "CUDA_VISIBLE_DEVICES": str(gpu)}, check=True)

with ThreadPoolExecutor(2) as pool:
    checks = [pool.submit(check, "pool_check", 0), pool.submit(check, "geometry_check", 1)]
    for result in checks:
        result.result()
print("[preflight] Both checks passed", flush=True)

## Обучение и инференс

In [ ]:
started = time.monotonic()
cmd = [*worker, "--output", str(WORK)]
log = ROOT / "suite.log"
with log.open("a", encoding="utf-8") as stream:
    with subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as job:
        for line in job.stdout:
            stream.write(line)
            print(line, end="", flush=True)
        if job.wait() != 0:
            stream.flush()
            shutil.copy2(log, WORK / "clevr_failed.log")
            print(f"Scratch and completed tasks are in {ROOT}. Rerun this cell in the same session.")
            raise subprocess.CalledProcessError(job.returncode, cmd)
with zipfile.ZipFile(WORK / "clevr_reports.zip", "a", compression=zipfile.ZIP_DEFLATED) as zip_report:
    zip_report.write(log, "suite.log")
print(f"Wall time including ZIP creation: {(time.monotonic() - started)/60:.1f} min")

In [ ]:
import pandas as pd
from IPython.display import FileLink, display

summary = pd.read_csv(ROOT / "result/reports/leaderboard.csv")
display(summary.style.format({"auc": "{:.8f}"}))
outputs = ["clevr_csv.zip", "clevr_reports.zip", "clevr_scores.zip", "clevr_weights.zip"]
os.chdir(WORK)
for name in outputs:
    path = WORK / name
    assert path.is_file()
    print(f"{name} ({path.stat().st_size / 2**20:.1f} MiB)")
    display(FileLink(name))